In [1]:
import pandas as pd
import re

In [8]:
def fuzzy_update_with_new_column(df_main, df_winman):

    efileds = {
        'Successfully e-verified',
        'ITR processed no demand no refund',
        'Refund Paid',
        'ITRV Received',
        'ITR processed refund determined and sent out to Refund Banker',
        'Refund kept on hold, Intimation u/s 245 is issued proposing adjustment of refund towards outstanding demand.',
        'ITR-V received after 30 days, thus date of verification is the date of filing of return.',
        'Successfully e-verified after 30 days, thus date of verification is the date of filing of return.'
        'Refund Failed at CPC', 
        'Refund failure'
    }

    # Helper function to normalize names into a set of words
    def get_name_parts(name):
        if not isinstance(name, str):
            return set()
        # Convert to lowercase and remove punctuation
        name = name.lower()
        name = re.sub(r'[^\w\s]', '', name)
        return set(name.split())

    # Create temporary columns with name parts for efficient matching
    df_main['_name_parts'] = df_main['Name'].apply(get_name_parts)
    df_winman['_name_parts'] = df_winman['Name'].apply(get_name_parts)

    update_log = []
    successful_matches = []
    ambiguous_matches = []
    no_matches = []

    # Iterate through the main dataframe that needs updating
    for _, row1 in df_main.iterrows():
        name_to_find = row1['Name']

        if row1['Status'] == 'E-Filed':
            update_log.append({
                'Original_Name': name_to_find,
                'from winman': pd.NA, # No update is performed
                'Comment': 'Skipped: Original status is already E-Filed'
            })
            continue


        parts_to_find = row1['_name_parts']
        
        if not parts_to_find:
            continue

        potential_matches = [
            row2 for _, row2 in df_winman.iterrows()
            if parts_to_find.issubset(row2['_name_parts'])
        ]

        # Case 1: Exactly one unique match found
        if len(potential_matches) == 1:
            matched_row = potential_matches[0]
            source_status = matched_row['Latest Status']

            if source_status in efileds:
                final_status = "E-Filed"
            else:
                final_status = source_status
                print(f"{final_status}")
            update_log.append({
                'Original_Name': name_to_find,
                'from winman': final_status,
                'Comment': f"Unique match: '{matched_row['Name']}'"
            })
            successful_matches.append({
                'Original_Name': name_to_find,
                'Original_Status': row1['Status'],
                'Matched_Name_in_Winman': matched_row['Name'],
                'Updated_Status_from_Winman': final_status
            })

        # Case 2: More than one match found (ambiguous)
        elif len(potential_matches) > 1:
            matched_names = [row['Name'] for row in potential_matches]
            update_log.append({
                'Original_Name': name_to_find,
                'from winman': pd.NA, # Use NA for ambiguous cases
                'Comment': f"Ambiguous: {len(potential_matches)} matches -> {matched_names}"
            })
            ambiguous_matches.append({'Original_Name': name_to_find, 'Potential_Matches': matched_names})

        # Case 3: No match found
        else:
            update_log.append({
                'Original_Name': name_to_find,
                'from winman': pd.NA, # Use NA for no match
                'Comment': "No match found in source"
            })
            no_matches.append({'Original_Name': name_to_find})

    # --- Create the final dataframe and insert columns ---
    df_final = df_main.copy()

    if not update_log: # Handle case where the loop was empty
        return df_final, pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    
    # Create a mapping from original name to the new data
    update_map = pd.DataFrame(update_log).set_index('Original_Name')
    
    # Map the new data into Series
    from_winman_series = df_final['Name'].map(update_map['from winman'])
    comment_series = df_final['Name'].map(update_map['Comment'])
    
    if 'from winman' in df_final.columns:
        # If it exists, overwrite the data
        df_final['from winman'] = from_winman_series
    else:
        # If not, insert it after the 'Status' column
        status_col_index = df_final.columns.get_loc('Status')
        df_final.insert(status_col_index + 1, 'from winman', from_winman_series)

    # Handle the 'Update_Comment' column
    if 'Update_Comment' in df_final.columns:
        # If it exists, overwrite the data
        df_final['Update_Comment'] = comment_series
    else:
        # If not, insert it after 'from winman'
        # We find the new index of 'from winman' to be safe
        winman_col_index = df_final.columns.get_loc('from winman')
        df_final.insert(winman_col_index + 1, 'Update_Comment', comment_series)
    
    # Clean up the temporary column
    df_final = df_final.drop(columns=['_name_parts'])

    # Create report dataframes
    report_matches = pd.DataFrame(successful_matches)
    report_ambiguous = pd.DataFrame(ambiguous_matches)
    report_no_match = pd.DataFrame(no_matches)

    return df_final, report_matches, report_ambiguous, report_no_match


# --- 3. Run the function and get the results ---



In [6]:
df1 = pd.read_excel("auto-imgs\\Winman 26-09-2025.xlsx")
df2 = pd.read_excel("auto-imgs\\Anirbaan - Copy of Updated ITR status.xlsx")

In [9]:
df1['Name'].str.contains('trust', case=False).sum()

np.int64(28)

In [10]:
df1["Latest Status"].unique()

array(['Successfully e-verified', 'ITR processed no demand no refund',
       'Pending for e-verification',
       'ITR processed refund determined and sent out to Refund Banker',
       'Refund Paid',
       'Refund kept on hold, Intimation u/s 245 is issued proposing adjustment of refund towards outstanding demand.',
       'ITRV Received',
       'Return uploaded, pending for ITR-V/ E-Verification',
       'Return discarded',
       'ITR-V received after 30 days, thus date of verification is the date of filing of return.',
       'Successfully e-verified after 30 days, thus date of verification is the date of filing of return.',
       'Refund Failed at CPC', 'Refund failure', 'Under Processing',
       'Processed with demand due',
       'ITR PROCESSED PARTIAL REFUND ADJUSTMENT DETERMINED'], dtype=object)

In [11]:
def clean_name(name):

    # Return original value if it's not a valid string
    if not isinstance(name, str) or not name.strip():
        return name
        
    # Remove common punctuation and split into parts
    # This helps with names like "Jones, Peter Michael"
    cleaned_name = re.sub(r'[,.]', '', name)
    parts = cleaned_name.split()
    
    # Apply the transformation logic
    if len(parts) >= 3:
        # Return the first and the last part, joined by a space
        return f"{parts[0]} {parts[-1]}"
    else:
        # Otherwise, return the name as is (joined back in case of extra spaces)
        return ' '.join(parts)

In [12]:
def process_names_in_dataframe(df):

    # It's good practice to work on a copy
    df_processed = df.copy()
    
    # Define the keywords to skip. The `|` means OR in regex.
    # `case=False` makes the search case-insensitive.
    # `na=False` treats NaN values as not containing the keywords.
    keywords_to_skip = r'trust|family|huf'
    mask_to_skip = df_processed['Name'].str.contains(keywords_to_skip, case=False, na=False)
    
    # We want to apply the cleaning function to rows that are NOT in the skip mask.
    # The `~` operator inverts the boolean mask (True becomes False, and vice-versa).
    mask_to_process = ~mask_to_skip
    
    # Use .loc to apply the clean_name function ONLY to the selected rows and the 'Name' column
    df_processed.loc[mask_to_process, 'Name'] = df_processed.loc[mask_to_process, 'Name'].apply(clean_name)
    
    return df_processed


In [14]:
df12 = process_names_in_dataframe(df1)

In [15]:
df_New_final, report_matches, report_ambiguous, report_no_match = fuzzy_update_with_new_column(df2, df12)

Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
Pending for e-verification
P

In [16]:
df_New_final.to_excel("New Updated ITR status.xlsx", engine="openpyxl")